# Stress Prediction V38 — Quantile/Weight Grid Search (다중 시드)

오늘 시도한 SVR, LightGBM, 도메인 피처, allostatic load, pair k-NN, FEATURE_COPY_COUNTS 재설계는
전부 v34 기준(CV 0.148 / Public 0.127)을 못 넘었습니다. v34는 이미 상당히 최적점 근처인 것 같아서,
마지막으로 **v34 핵심 로직(quantile 집계 + 블렌드) 자체의 하이퍼파라미터**를 체계적으로 재탐색합니다.

**핵심 아이디어**: `TREE_QUANTILE`, `PAIR_QUANTILE`, 블렌드 가중치는 전부 이미 학습된 모델의
예측값에 대한 사후 집계 방식일 뿐이라, 조합마다 모델을 다시 학습할 필요가 없습니다.
- 트리 모델 학습 -> 개별 트리 1200개의 예측값 행렬 저장 (quantile과 무관, 한 번만 계산)
- Pair 방식 -> 각 피처쌍의 최근접 이웃 타겟값 행렬 저장 (quantile과 무관, 한 번만 계산)
- 이후 quantile/weight 조합은 이 행렬에 `np.quantile`만 적용하면 되므로 매우 빠릅니다

**다중 시드**: 5-fold를 서로 다른 시드로 3번 반복(총 15개 검증셋)해서, 특정 분할에만 맞는 조합을
고르지 않도록 했습니다. v34가 34번의 손튜닝 끝에 찾은 값(0.54/0.48/0.28)이 정말 최적인지,
아니면 더 나은 조합이 있는지 확인합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations, product
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"

# 검색용 트리 개수 (탐색 속도 위해 축소). 최적 조합 찾은 뒤 최종 제출은 1200으로 재확인 권장.
N_ESTIMATORS_SEARCH = 600
N_SEED_REPEATS = 3       # 5-fold를 몇 번 다른 시드로 반복할지
SEEDS = [42, 7, 2026][:N_SEED_REPEATS]

TREE_Q_GRID = [0.50, 0.52, 0.54, 0.56, 0.58]
PAIR_Q_GRID = [0.44, 0.46, 0.48, 0.50, 0.52]
WEIGHT_GRID = [0.20, 0.24, 0.28, 0.32, 0.36]

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
    "age",
]


In [2]:

def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """V34 원본과 동일 (행 단위, leakage 없음)."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_predictions_matrix(train_features, val_features, target, seed) -> np.ndarray:
    """개별 트리 예측값 행렬 (n_val x n_estimators). quantile과 무관하게 한 번만 계산."""
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS_SEARCH,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)
    return np.column_stack([t.predict(val_matrix) for t in model.estimators_])


def empirical_rank_transform(train_values: np.ndarray, test_values: np.ndarray):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_matrix(train_features, val_features, target) -> np.ndarray:
    """각 피처쌍의 최근접이웃 타겟값 행렬 (n_val x n_pairs). quantile과 무관하게 한 번만 계산."""
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.column_stack(neighbor_targets)


## Grid Search 실행

시드 3개 x 5-fold = 15개 검증셋에서, 트리/페어 예측 행렬을 한 번씩만 계산하고
125개(5x5x5) quantile/weight 조합을 전부 사후 평가합니다.


In [3]:

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
train = pd.read_csv(DATA_DIR / "train.csv")

raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

grid_combos = list(product(TREE_Q_GRID, PAIR_Q_GRID, WEIGHT_GRID))
scores = {combo: [] for combo in grid_combos}

split_count = 0
for seed in SEEDS:
    cv = KFold(5, shuffle=True, random_state=seed)
    for train_idx, val_idx in cv.split(raw_X):
        split_count += 1
        train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
        val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].to_numpy(dtype=float)

        tree_matrix = tree_predictions_matrix(train_feat, val_feat, y_train, seed)
        pair_matrix = pair_neighbor_matrix(train_feat, val_feat, y_train)

        # quantile별 예측을 미리 다 계산해두고 조합에서는 조회만
        tree_by_q = {tq: np.quantile(tree_matrix, tq, axis=1) for tq in TREE_Q_GRID}
        pair_by_q = {pq: np.quantile(pair_matrix, pq, axis=1) for pq in PAIR_Q_GRID}

        for tq, pq, w in grid_combos:
            combined = np.clip(np.round((1 - w) * tree_by_q[tq] + w * pair_by_q[pq], 2), 0.0, 1.0)
            scores[(tq, pq, w)].append(mean_absolute_error(y_val, combined))

        print(f"split {split_count}/{len(SEEDS) * 5} done")

results = pd.DataFrame([
    {"tree_quantile": tq, "pair_quantile": pq, "pair_weight": w,
     "mean_MAE": np.mean(v), "std_MAE": np.std(v)}
    for (tq, pq, w), v in scores.items()
]).sort_values("mean_MAE").reset_index(drop=True)

print(f"총 {len(SEEDS)}개 시드 x 5-fold = {len(SEEDS) * 5}개 검증셋, {len(grid_combos)}개 조합 탐색 완료")
print()
print("현재 v34 원본 설정(0.54, 0.48, 0.28)의 성적:")
current = results[
    (results.tree_quantile == 0.54) & (results.pair_quantile == 0.48) & (results.pair_weight == 0.28)
]
display(current)
print()
print("상위 15개 조합:")
display(results.head(15))


split 1/15 done
split 2/15 done
split 3/15 done
split 4/15 done
split 5/15 done
split 6/15 done
split 7/15 done
split 8/15 done
split 9/15 done
split 10/15 done
split 11/15 done
split 12/15 done
split 13/15 done
split 14/15 done
split 15/15 done
총 3개 시드 x 5-fold = 15개 검증셋, 125개 조합 탐색 완료

현재 v34 원본 설정(0.54, 0.48, 0.28)의 성적:


,tree_quantile,pair_quantile,pair_weight,mean_MAE,std_MAE
39,0.54,0.48,0.28,0.146856,0.005229



상위 15개 조합:


,tree_quantile,pair_quantile,pair_weight,mean_MAE,std_MAE
0,0.50,0.52,0.24,0.146631,0.005262
1,0.50,0.52,0.20,0.146658,0.005339
2,0.50,0.52,0.28,0.146686,0.005266
3,0.52,0.50,0.28,0.146701,0.005245
4,0.50,0.52,0.32,0.146707,0.005278
5,0.52,0.52,0.24,0.146717,0.005314
6,0.52,0.48,0.24,0.146719,0.005238
7,0.52,0.50,0.20,0.146720,0.005349
8,0.52,0.50,0.24,0.146737,0.005272
9,0.52,0.52,0.32,0.146739,0.005300


## 최적 조합 확인 후

위 결과에서 `mean_MAE`가 가장 낮은 조합을 찾았으면, 원본 v34 노트북(또는 v36/v37 ablation 노트북)의
`TREE_QUANTILE`, `PAIR_QUANTILE`, 블렌드 가중치(`0.72*tree+0.28*pair`의 0.28 부분)를 그 값으로 바꾸고,
`N_ESTIMATORS_SEARCH=600` 대신 원래 `n_estimators=1200`으로 최종 제출 파일을 다시 생성해서 검증하세요
(트리 개수를 줄여서 탐색했기 때문에, 최종 확인은 1200개로 하는 게 안전합니다).
